# 07 — Backtest & Walk-Forward Validation

**Purpose:** Run a realistic event-driven backtest on strategy candidates discovered in previous notebooks.

**What is tested:**
- Entry signals from the best ML model
- Realistic costs: spread + slippage + commission
- TP/SL/max-hold exits
- Session time filter (no trades outside DAX hours)
- One-position-at-a-time enforcement
- Out-of-sample walk-forward: each fold trained on past data, tested on future data only

**Metrics reported:**
- Net profit, win rate, profit factor, expectancy, average R
- Sharpe ratio, Sortino ratio (annualised)
- Maximum drawdown (absolute + %)
- Monthly P&L breakdown
- Trade log

**Inputs:** `data/features/`, `models/`  
**Outputs:** `reports/backtest_*.csv`, `reports/equity_curve.png`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
from pathlib import Path

from src.utils.config import (
    load_config, get_symbol, get_paths, get_backtest_params, get_validation_params
)
from src.models.train_ml import build_models, walk_forward_splits
from src.backtest.event_backtester import EventBacktester
from sklearn.base import clone

cfg      = load_config()
paths    = get_paths(cfg, "..")
symbol   = get_symbol(cfg)
bt_cfg   = get_backtest_params(cfg)
val_cfg  = get_validation_params(cfg)

print("Backtest config:")
for k, v in bt_cfg.items():
    print(f"  {k}: {v}")

## 1. Load data and prepare

In [ ]:
TP_ATR, SL_ATR, HORIZON = 2.0, 1.0, 30
key     = f"tp{TP_ATR}_sl{SL_ATR}_h{HORIZON}"
labeled = pd.read_parquet(paths["features"] / f"{symbol}_labeled_{key}.parquet")

exclude_prefixes = (
    "label_", "barrier_side_", "time_to_exit_", "mfe_", "mae_"
)
ohlcv_cols    = ["open", "high", "low", "close", "tick_volume"]
feature_cols  = [
    c for c in labeled.columns
    if c not in ohlcv_cols
    and not any(c.startswith(p) for p in exclude_prefixes)
]

prices  = labeled[ohlcv_cols]
X       = labeled[feature_cols].fillna(0)
y_long  = labeled["label_long"]
atr     = labeled["atr_14"]

valid_mask = labeled[feature_cols].notna().all(axis=1)
prices  = prices[valid_mask]
X       = X[valid_mask]
y_long  = y_long[valid_mask]
atr     = atr[valid_mask]

print(f"Dataset: {len(X)} bars, {len(feature_cols)} features")

## 2. Walk-forward backtest

For each walk-forward fold:
1. Train model on `train_idx` (chronologically earlier)
2. Generate predictions on `test_idx` (future, unseen data)
3. Run event-driven backtest on those predictions
4. Collect metrics per fold

In [ ]:
splits = walk_forward_splits(
    n=len(X),
    n_splits=val_cfg["n_splits"],
    train_pct=val_cfg["train_pct"],
    purge_bars=val_cfg["purge_bars"],
    embargo_bars=val_cfg["embargo_bars"],
)

all_models = build_models()
MODEL_NAME = "random_forest"  # change to any model from build_models()
model      = all_models[MODEL_NAME]

fold_results   = []
all_signals    = pd.Series(0, index=X.index, dtype=int)

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    Xtr, ytr = X.iloc[train_idx], y_long.iloc[train_idx]
    Xte       = X.iloc[test_idx]

    train_mask = ytr != 0
    if train_mask.sum() < 10:
        print(f"Fold {fold_idx}: skipped (too few signal bars)")
        continue

    m = clone(model)
    m.fit(Xtr.values[train_mask], ytr.values[train_mask])
    y_pred = m.predict(Xte.values)

    # Only act on predicted signal bars (1 = long, -1 = short, 0 = flat)
    test_timestamps = X.index[test_idx]
    signals_fold    = pd.Series(y_pred, index=test_timestamps)
    all_signals.loc[test_timestamps] = y_pred

    # Backtest this fold
    prices_fold = prices.iloc[test_idx]
    atr_fold    = atr.iloc[test_idx]

    bt = EventBacktester(
        prices=prices_fold,
        signals=signals_fold,
        cfg_backtest=bt_cfg,
        atr=atr_fold,
        tp_atr=TP_ATR,
        sl_atr=SL_ATR,
    )
    result = bt.run()

    m_dict = result.metrics.copy()
    m_dict["fold"]       = fold_idx
    m_dict["start_date"] = str(prices_fold.index[0].date())
    m_dict["end_date"]   = str(prices_fold.index[-1].date())
    fold_results.append(m_dict)

    print(
        f"Fold {fold_idx} ({m_dict['start_date']} → {m_dict['end_date']}): "
        f"trades={m_dict.get('n_trades',0):3d} "
        f"net={m_dict.get('net_profit',0):+.1f} "
        f"pf={m_dict.get('profit_factor',0):.2f} "
        f"wr={m_dict.get('win_rate',0)*100:.1f}%"
    )

fold_df = pd.DataFrame(fold_results)
print("\nWalk-forward summary:")
display(fold_df.set_index("fold"))

## 3. Full out-of-sample backtest (concatenated folds)

In [ ]:
# Run backtest on all walk-forward signals combined
oos_mask = all_signals != 0

bt_full = EventBacktester(
    prices=prices,
    signals=all_signals,
    cfg_backtest=bt_cfg,
    atr=atr,
    tp_atr=TP_ATR,
    sl_atr=SL_ATR,
)
result_full = bt_full.run()

print("Full OOS backtest metrics:")
for k, v in result_full.metrics.items():
    print(f"  {k:22s}: {v:.4f}" if isinstance(v, float) else f"  {k:22s}: {v}")

## 4. Equity curve & drawdown

In [ ]:
equity = result_full.equity_curve.cumsum()
drawdown = equity - equity.cummax()

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1, 1])

ax0 = fig.add_subplot(gs[0])
ax0.plot(equity.index, equity.values, lw=1.2, color="steelblue")
ax0.axhline(0, color="grey", lw=0.5, ls="--")
ax0.set_title(f"{symbol} — Walk-Forward Equity Curve ({MODEL_NAME})")
ax0.set_ylabel("Cumulative Net P&L (EUR)")
ax0.grid(alpha=0.3)

ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax1.fill_between(drawdown.index, drawdown.values, 0, alpha=0.6, color="red")
ax1.set_ylabel("Drawdown (EUR)")
ax1.grid(alpha=0.3)

# Monthly P&L bar chart
ax2 = fig.add_subplot(gs[2])
monthly = result_full.monthly_stats
if not monthly.empty:
    colors = ["green" if p >= 0 else "red" for p in monthly["net_pnl"]]
    ax2.bar(monthly["month"].astype(str), monthly["net_pnl"], color=colors, alpha=0.8)
    ax2.set_ylabel("Monthly P&L")
    ax2.tick_params(axis="x", rotation=45)
    ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/equity_curve.png", dpi=120)
plt.show()

## 5. Save trade log and monthly stats

In [ ]:
reports_dir = paths["reports"]
reports_dir.mkdir(parents=True, exist_ok=True)

# Trade log
if not result_full.trades.empty:
    trade_log_path = reports_dir / f"backtest_trades_{MODEL_NAME}_{key}.csv"
    result_full.trades.to_csv(trade_log_path)
    print(f"Trade log saved → {trade_log_path}")
    print(f"Total trades: {len(result_full.trades)}")
    display(result_full.trades.head(10))

# Monthly stats
if not result_full.monthly_stats.empty:
    monthly_path = reports_dir / f"backtest_monthly_{MODEL_NAME}_{key}.csv"
    result_full.monthly_stats.to_csv(monthly_path, index=False)
    print(f"Monthly stats saved → {monthly_path}")
    display(result_full.monthly_stats)

# Walk-forward fold summary
wf_path = reports_dir / f"backtest_walkforward_{MODEL_NAME}_{key}.csv"
fold_df.to_csv(wf_path, index=False)
print(f"Walk-forward summary saved → {wf_path}")